# SPY 20-min Candle Trigger Strategy

**Entry signal** (each trading day):
- Build 20-min OHLC candles anchored at 9:30 ET: [9:30, 9:50), [9:50, 10:10), [10:10, 10:30), …
- A candle is **green** if `close > open`, **red** if `close < open`. Doji (`close == open`) breaks any streak.
- On the **first** occurrence of two consecutive same-direction candles, fire:
  - 2 greens → buy **ATM call**
  - 2 reds → buy **ATM put**
- Entry fills at the **OPEN of the next 1-min candle** after the 2nd 20-min candle closes (e.g. 2nd candle 10:10–10:30 → entry at 10:30 open).
- **Latest entry time: 12:30 ET (9:30 PT).** If the 2nd candle closes after that, skip the day.

**Position:**
- ATM strike = nearest $1 to spot (round half up).
- DTE swept as a dimension: **0 DTE** (today's expiry) vs **2 DTE** (today + 2 business days, weekends crossed).

**Exits (swept):**
- Profit target: 3% gross, 3% net (after fees), 5% gross, 5% net, 10% gross.
- Stop loss: 15%, 20%, 25%, 30% (mode pairs with PT's mode).
- Hard close at 3:55 PM ET fallback.

**Sweep grid:** 2 DTE × 5 PT × 4 SL = **40 configs**.

**Account:** $1,500 start, $50k max-per-trade, $0.85/contract commission, $0.01 half-spread.


## 1. Setup

In [ ]:
import os, sys
REPO = '/content/iron-condor'
BRANCH = 'claude/spy-options-trading-bot-ri4Ms'
if not os.path.isdir(REPO):
    !git clone --quiet -b {BRANCH} https://github.com/coolin815/iron-condor.git {REPO}
else:
    !cd {REPO} && git pull --quiet
SRC = REPO + '/src'
os.environ['PYTHONPATH'] = SRC + os.pathsep + os.environ.get('PYTHONPATH', '')
if SRC not in sys.path:
    sys.path.insert(0, SRC)
os.chdir(REPO)
print('OK')

## 2. Paste your Polygon key

In [ ]:
import os, getpass
os.environ['POLYGON_API_KEY'] = getpass.getpass('Polygon API key: ')
print('Key loaded:', len(os.environ['POLYGON_API_KEY']), 'chars')

## 3. Smoke test — one recent day
Pipeline sanity check. Output is a single TradeResult (or `no_signal`/`no_data`).

In [ ]:
!python -m iron_condor.cli --smoke -v

## 4. Sweep — past 7 days
~1–2 min. ~5 trading days. Pipeline check + first read.

In [ ]:
!python -m iron_condor.cli --days 7 --sweep

## 4b. Sweep — past 30 days
~5 min. ~21 trading days. First real read on edge.

In [ ]:
!python -m iron_condor.cli --days 30 --sweep

## 4c. Sweep — past 60 days
~10 min. ~42 trading days.

In [ ]:
!python -m iron_condor.cli --days 60 --sweep

## 4d. Sweep — past 365 days
Full year. Run only if shorter windows looked promising.

In [ ]:
!python -m iron_condor.cli --days 365 --sweep

## 5. Top configs
Reads `results/sweep_summary.csv` from whichever sweep cell you last ran.

In [ ]:
import pandas as pd
summary = pd.read_csv('results/sweep_summary.csv')
summary.head(40)

## 6. DTE comparison
0 DTE vs 2 DTE — which expiration captures the signal better?

In [ ]:
import re
rows = []
for _, r in summary.iterrows():
    m = re.match(r'd(\d+)\|pt(\d+)([gn])\|sl(\d+)', r['config'])
    if not m: continue
    rows.append({
        'dte': int(m.group(1)),
        'pt_label': f'{m.group(2)}%{m.group(3)}',
        'sl_label': f'{m.group(4)}%',
        'return': r['total_return_pct'],
        'win_rate': r['win_rate'],
        'trades': r['n_trades'],
        'avg_pnl': r['avg_net_pnl'],
    })
grid = pd.DataFrame(rows)

print('=== Avg return % by DTE ===')
print(grid.groupby('dte')['return'].agg(['mean', 'median', 'max']).round(1))
print('\n=== Avg WR by DTE ===')
print(grid.groupby('dte')['win_rate'].agg(['mean', 'median', 'max']).round(2))
print('\n=== Avg per-trade $ by DTE ===')
print(grid.groupby('dte')['avg_pnl'].agg(['mean', 'median', 'min', 'max']).round(2))

## 7. PT × SL heatmap (per DTE)
Total return by profit-target and stop-loss config, split by DTE.

In [ ]:
for dte in sorted(grid['dte'].unique()):
    sub = grid[grid['dte'] == dte]
    print(f'\n=== DTE={dte} — Total return % by (PT, SL) ===')
    print(sub.pivot(index='pt_label', columns='sl_label', values='return').round(1))

## 8. Top config breakdown — call vs put, exit reasons

In [ ]:
trades = pd.read_csv('results/sweep_trades.csv')
top_cfg = summary.iloc[0]['config']
sub = trades[trades['config'] == top_cfg]
taken = sub[sub['exit_reason'].isin(['profit', 'stop', 'hard_close'])]
print(f'Top config: {top_cfg}')
print(f'Total days: {len(sub)}, taken trades: {len(taken)}')
print('\n=== Call vs put performance ===')
if not taken.empty and 'right' in taken.columns:
    print(taken.groupby('right').agg(
        trades=('net_pnl', 'count'),
        win_rate=('net_pnl', lambda s: (s > 0).mean()),
        avg_pnl=('net_pnl', 'mean'),
        total_pnl=('net_pnl', 'sum'),
    ).round(2))
print('\n=== Exit reasons ===')
print(sub['exit_reason'].value_counts())
print('\n=== Hold times (minutes) ===')
if not taken.empty and 'minutes_held' in taken.columns:
    print(taken['minutes_held'].describe().round(1))

## 9. Equity curve — top 3 configs

In [ ]:
import matplotlib.pyplot as plt
top3 = summary.head(3)['config'].tolist()
fig, ax = plt.subplots(figsize=(11, 5))
for cfg in top3:
    s = trades[trades['config'] == cfg].sort_values('day')
    ax.plot(pd.to_datetime(s['day']), s['balance_after'], label=cfg, linewidth=1.5)
ax.axhline(1500, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Date'); ax.set_ylabel('Balance ($)')
ax.set_title('20-min candle trigger — top 3 equity curves')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 10. Per-month breakdown
Splits the trade log by calendar month to see if the result concentrates in 1–2 months.

In [ ]:
trades_all = pd.read_csv('results/sweep_trades.csv')
trades_all['day'] = pd.to_datetime(trades_all['day'])
trades_all['month'] = trades_all['day'].dt.to_period('M').astype(str)
taken_all = trades_all[trades_all['exit_reason'].isin(['profit', 'stop', 'hard_close'])]

print('=== Per-month aggregate (all configs averaged) ===')
print(taken_all.groupby('month').agg(
    trades=('net_pnl', 'count'),
    win_rate=('net_pnl', lambda s: (s > 0).mean()),
    avg_pnl=('net_pnl', 'mean'),
    total_pnl=('net_pnl', 'sum'),
).round(2).to_string())

top_cfg = summary.iloc[0]['config']
top_taken = taken_all[taken_all['config'] == top_cfg]
if not top_taken.empty:
    print(f'\n=== Top config ({top_cfg}) per month ===')
    print(top_taken.groupby('month').agg(
        trades=('net_pnl', 'count'),
        win_rate=('net_pnl', lambda s: (s > 0).mean()),
        avg_pnl=('net_pnl', 'mean'),
        total_pnl=('net_pnl', 'sum'),
    ).round(2).to_string())